
# CPU-only model optimization experiment for Smart Transaction Categorization

This notebook adapts the **model optimizations for serving** lab to your project.

## What this notebook does
It builds a **CPU-only model-level benchmark harness** for four candidate model artifacts:

1. **PyTorch eager** baseline  
2. **PyTorch compiled** candidate (`torch.compile`)  
3. **ONNX Runtime CPU** candidate  
4. **ONNX Runtime dynamic quantized** candidate  

## Important project-specific note
Your proposal says the most likely production serving path is a **lightweight CPU classifier** on a single Chameleon VM. This notebook is therefore designed as a **model-level shortlist experiment**, not a final deployment decision.

To make `eager` / `compiled` / `onnx` / `quantization` comparable in one place, this notebook trains a **small surrogate PyTorch classifier** on representative synthetic transaction data. That is useful for model-level benchmarking because the serving notes recommend evaluating **the model artifact itself in isolation on fixed hardware with identical inputs** before mixing in system-level effects.

## How to interpret the result
- If `compiled`, `onnx`, or `quantized` wins on latency / throughput / size **without unacceptable quality loss**, you can shortlist it for the next serving stage.
- If the gains are small, that is still a valid result. For your real project, that would support the argument that the simpler CPU baseline remains the best overall choice.



## Metrics collected
For each variant, this notebook records:

- top-1 accuracy on a held-out synthetic test set
- top-3 hit rate on the held-out synthetic test set
- p50 latency for **batch size 1**
- p95 latency for **batch size 1**
- throughput for **batch size 32**
- artifact size on disk

This matches the serving notes' recommendation to compare model variants in a controlled harness with fixed hardware, identical inputs, and explicit speed/quality tradeoffs.


In [ ]:

# Core imports.
# Keep all imports near the top so the notebook can be executed end-to-end on Chameleon.

import json
import math
import os
import random
import string
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

# Make notebook runs repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# File locations inside the repo.
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODELS_DIR = ROOT / 'models'
RESULTS_DIR = ROOT / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('MODELS_DIR =', MODELS_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('Torch version =', torch.__version__)
print('ONNX Runtime version =', ort.__version__)


## 1. Create representative synthetic transaction data

In [ ]:

# This synthetic generator is intentionally simple but representative.
# It mirrors your project's input shape:
# - merchant text / payee text
# - amount
# - currency
# - transaction date features
# - account type
# and it produces category labels like your Actual category_ids.

CATEGORY_PATTERNS = {
    'groceries': ['WHOLE FOODS', 'TRADER JOES', 'KROGER', 'SAFEWAY', 'COSTCO'],
    'restaurants': ['UBER EATS', 'DOORDASH', 'MCDONALDS', 'CHIPOTLE', 'STARBUCKS'],
    'transportation': ['UBER TRIP', 'LYFT', 'MTA', 'SHELL', 'CHEVRON'],
    'rent': ['LANDLORD LLC', 'APARTMENT RENT', 'PROPERTY MGMT'],
    'utilities': ['COMCAST', 'VERIZON', 'AT&T', 'DUKE ENERGY'],
    'shopping': ['AMZN MKTPLACE', 'TARGET', 'WALMART', 'BEST BUY'],
    'entertainment': ['NETFLIX', 'SPOTIFY', 'AMC THEATRES', 'STEAM PURCHASE'],
    'travel': ['DELTA AIR', 'UNITED', 'HILTON', 'MARRIOTT', 'AIRBNB'],
}

ACCOUNT_TYPES = ['checking', 'credit', 'cash']
CURRENCIES = ['USD']

AMOUNT_RANGES = {
    'groceries': (15, 180),
    'restaurants': (8, 70),
    'transportation': (3, 90),
    'rent': (900, 2800),
    'utilities': (30, 220),
    'shopping': (10, 500),
    'entertainment': (5, 120),
    'travel': (60, 1200),
}

ACCOUNT_PRIORS = {
    'groceries': ['credit', 'checking'],
    'restaurants': ['credit'],
    'transportation': ['credit', 'checking'],
    'rent': ['checking'],
    'utilities': ['checking', 'credit'],
    'shopping': ['credit'],
    'entertainment': ['credit'],
    'travel': ['credit'],
}

MONTH_PRIORS = {
    'travel': [5, 6, 7, 11, 12],
    'rent': list(range(1, 13)),
    'utilities': list(range(1, 13)),
}

SUFFIXES = ['1234', 'ONLINE', 'HELP', 'PMTS', 'STORE', 'NY', 'CA', 'REF']


def random_description(base: str) -> str:
    """Add realistic noise and formatting variation to merchant strings."""
    templates = [
        '{base}',
        '{base} {suffix}',
        '{base} * {suffix}',
        '{base} #{num}',
        '{base} {num}',
    ]
    template = random.choice(templates)
    return template.format(base=base, suffix=random.choice(SUFFIXES), num=random.randint(100, 9999))


def sample_month(category: str) -> int:
    if category in MONTH_PRIORS and random.random() < 0.7:
        return random.choice(MONTH_PRIORS[category])
    return random.randint(1, 12)


def generate_example(category: str) -> dict:
    low, high = AMOUNT_RANGES[category]
    base = random.choice(CATEGORY_PATTERNS[category])
    amount = round(random.uniform(low, high), 2)
    month = sample_month(category)
    day = random.randint(1, 28)
    account_type = random.choice(ACCOUNT_PRIORS.get(category, ACCOUNT_TYPES))
    return {
        'merchant_text': random_description(base),
        'amount': amount,
        'currency': random.choice(CURRENCIES),
        'transaction_date': f'2025-{month:02d}-{day:02d}',
        'account_type': account_type,
        'category_id': category,
    }


def build_dataset(n_per_class: int = 700) -> pd.DataFrame:
    rows = []
    for category in CATEGORY_PATTERNS:
        for _ in range(n_per_class):
            rows.append(generate_example(category))
    df = pd.DataFrame(rows).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return df


df = build_dataset(n_per_class=700)
print(df.head())
print('
Rows:', len(df))
print('
Class counts:')
print(df['category_id'].value_counts())


## 2. Build numeric features outside the model artifact

In [ ]:

# This mirrors a realistic serving design where preprocessing can stay separate
# from the core model artifact. For model-only benchmarking, we freeze inputs as
# numeric tensors so that eager / compiled / ONNX / quantized variants are compared fairly.


def add_time_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    dt = pd.to_datetime(out['transaction_date'])
    out['month'] = dt.dt.month.astype(np.int64)
    out['day'] = dt.dt.day.astype(np.int64)
    return out


df = add_time_features(df)
label_list = sorted(df['category_id'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(label_list)}
y = df['category_id'].map(label_to_idx).astype(np.int64).values

train_df, test_df, y_train, y_test = train_test_split(
    df, y, test_size=0.2, random_state=SEED, stratify=y
)

# Text features.
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=2,
    max_features=2048,
)
X_text_train = vectorizer.fit_transform(train_df['merchant_text'])
X_text_test = vectorizer.transform(test_df['merchant_text'])

# Numeric features.
num_cols = ['amount', 'month', 'day']
scaler = StandardScaler()
X_num_train = scaler.fit_transform(train_df[num_cols])
X_num_test = scaler.transform(test_df[num_cols])

# Categorical metadata.
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat_train = ohe.fit_transform(train_df[['account_type', 'currency']])
X_cat_test = ohe.transform(test_df[['account_type', 'currency']])

# Final dense feature matrices for the surrogate neural model.
X_train = np.hstack([X_text_train.toarray().astype(np.float32), X_num_train.astype(np.float32), X_cat_train.astype(np.float32)])
X_test = np.hstack([X_text_test.toarray().astype(np.float32), X_num_test.astype(np.float32), X_cat_test.astype(np.float32)])

print('X_train shape =', X_train.shape)
print('X_test shape =', X_test.shape)
print('Num labels =', len(label_list))

feature_meta = {
    'vectorizer_max_features': 2048,
    'num_features': int(X_train.shape[1]),
    'labels': label_list,
}
with open(MODELS_DIR / 'surrogate_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)


## 3. Train a small surrogate PyTorch classifier

In [ ]:

# The goal is not to beat your real training team model.
# The goal is to produce a stable, CPU-friendly artifact that allows one notebook
# to compare eager / compiled / ONNX / quantized variants in the style of the lab.

class TxClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool):
    x_tensor = torch.from_numpy(X)
    y_tensor = torch.from_numpy(y)
    dataset = torch.utils.data.TensorDataset(x_tensor, y_tensor)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, batch_size=64, shuffle=True)
test_loader = make_loader(X_test, y_test, batch_size=256, shuffle=False)

device = torch.device('cpu')
model = TxClassifier(input_dim=X_train.shape[1], hidden_dim=128, num_classes=len(label_list)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 8
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch {epoch + 1}/{EPOCHS} - loss={epoch_loss:.4f}')

fp32_torch_path = MODELS_DIR / 'tx_surrogate_fp32.pt'
torch.save(model.state_dict(), fp32_torch_path)
print('Saved torch model to', fp32_torch_path)


## 4. Shared evaluation helpers

In [ ]:

# These helpers are intentionally verbose and commented.
# They are meant to be reused later when you adapt this notebook on Chameleon.


def evaluate_torch_model(torch_model: nn.Module, X_eval: np.ndarray, y_eval: np.ndarray):
    torch_model.eval()
    with torch.no_grad():
        logits = torch_model(torch.from_numpy(X_eval)).numpy()
    top1 = accuracy_score(y_eval, np.argmax(logits, axis=1))
    top3 = top_k_accuracy_score(y_eval, logits, k=min(3, logits.shape[1]), labels=list(range(logits.shape[1])))
    return {'top1_accuracy': float(top1), 'top3_hit_rate': float(top3)}


def evaluate_onnx_session(session: ort.InferenceSession, X_eval: np.ndarray, y_eval: np.ndarray):
    input_name = session.get_inputs()[0].name
    logits = session.run(None, {input_name: X_eval.astype(np.float32)})[0]
    top1 = accuracy_score(y_eval, np.argmax(logits, axis=1))
    top3 = top_k_accuracy_score(y_eval, logits, k=min(3, logits.shape[1]), labels=list(range(logits.shape[1])))
    return {'top1_accuracy': float(top1), 'top3_hit_rate': float(top3)}


def benchmark_torch_latency(torch_model: nn.Module, sample: np.ndarray, num_trials: int = 200):
    torch_model.eval()
    x = torch.from_numpy(sample)
    # Warm-up so lazy initialization does not distort the measurement.
    with torch.no_grad():
        for _ in range(20):
            _ = torch_model(x)
    latencies_ms = []
    with torch.no_grad():
        for _ in range(num_trials):
            start = time.perf_counter()
            _ = torch_model(x)
            end = time.perf_counter()
            latencies_ms.append((end - start) * 1000)
    return {
        'p50_latency_ms': float(np.percentile(latencies_ms, 50)),
        'p95_latency_ms': float(np.percentile(latencies_ms, 95)),
    }


def benchmark_torch_throughput(torch_model: nn.Module, batch: np.ndarray, num_trials: int = 200):
    torch_model.eval()
    x = torch.from_numpy(batch)
    with torch.no_grad():
        for _ in range(20):
            _ = torch_model(x)
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_trials):
            _ = torch_model(x)
    end = time.perf_counter()
    total_examples = batch.shape[0] * num_trials
    return {'throughput_items_per_sec': float(total_examples / (end - start))}


def benchmark_onnx_latency(session: ort.InferenceSession, sample: np.ndarray, num_trials: int = 200):
    input_name = session.get_inputs()[0].name
    for _ in range(20):
        _ = session.run(None, {input_name: sample.astype(np.float32)})
    latencies_ms = []
    for _ in range(num_trials):
        start = time.perf_counter()
        _ = session.run(None, {input_name: sample.astype(np.float32)})
        end = time.perf_counter()
        latencies_ms.append((end - start) * 1000)
    return {
        'p50_latency_ms': float(np.percentile(latencies_ms, 50)),
        'p95_latency_ms': float(np.percentile(latencies_ms, 95)),
    }


def benchmark_onnx_throughput(session: ort.InferenceSession, batch: np.ndarray, num_trials: int = 200):
    input_name = session.get_inputs()[0].name
    for _ in range(20):
        _ = session.run(None, {input_name: batch.astype(np.float32)})
    start = time.perf_counter()
    for _ in range(num_trials):
        _ = session.run(None, {input_name: batch.astype(np.float32)})
    end = time.perf_counter()
    total_examples = batch.shape[0] * num_trials
    return {'throughput_items_per_sec': float(total_examples / (end - start))}


def file_size_mb(path: Path) -> float:
    return float(path.stat().st_size / 1e6)

single_sample = X_test[:1].astype(np.float32)
batch_32 = X_test[:32].astype(np.float32)


## 5. Variant A — PyTorch eager baseline

In [ ]:

# Reload from disk to make the baseline artifact explicit.
eager_model = TxClassifier(input_dim=X_train.shape[1], hidden_dim=128, num_classes=len(label_list))
eager_model.load_state_dict(torch.load(fp32_torch_path, map_location='cpu'))
eager_model.eval()

eager_metrics = {}
eager_metrics.update(evaluate_torch_model(eager_model, X_test.astype(np.float32), y_test))
eager_metrics.update(benchmark_torch_latency(eager_model, single_sample))
eager_metrics.update(benchmark_torch_throughput(eager_model, batch_32))
eager_metrics['artifact_size_mb'] = file_size_mb(fp32_torch_path)
eager_metrics['variant'] = 'torch_eager_fp32'
print(json.dumps(eager_metrics, indent=2))


## 6. Variant B — PyTorch compiled candidate

In [ ]:

# torch.compile is meaningful here because the artifact is a PyTorch model.
# On some CPU environments the gain may be small or the compile may fail.
# That is still a valid experimental outcome.

compiled_metrics = {'variant': 'torch_compiled_fp32'}
compiled_model = None
try:
    base_for_compile = TxClassifier(input_dim=X_train.shape[1], hidden_dim=128, num_classes=len(label_list))
    base_for_compile.load_state_dict(torch.load(fp32_torch_path, map_location='cpu'))
    base_for_compile.eval()

    if hasattr(torch, 'compile'):
        compiled_model = torch.compile(base_for_compile)
        # Trigger compilation before timing.
        with torch.no_grad():
            _ = compiled_model(torch.from_numpy(single_sample))
        compiled_metrics.update(evaluate_torch_model(compiled_model, X_test.astype(np.float32), y_test))
        compiled_metrics.update(benchmark_torch_latency(compiled_model, single_sample))
        compiled_metrics.update(benchmark_torch_throughput(compiled_model, batch_32))
        compiled_metrics['artifact_size_mb'] = file_size_mb(fp32_torch_path)
        compiled_metrics['status'] = 'ok'
    else:
        compiled_metrics['status'] = 'torch_compile_unavailable'
except Exception as e:
    compiled_metrics['status'] = f'failed: {type(e).__name__}: {e}'

print(json.dumps(compiled_metrics, indent=2))


## 7. Variant C — ONNX Runtime CPU candidate

In [ ]:

# Export a plain fp32 ONNX graph. This is closest to the lab's ONNX path,
# but adapted to your transaction classifier surrogate.

onnx_path = MODELS_DIR / 'tx_surrogate_fp32.onnx'
export_model = TxClassifier(input_dim=X_train.shape[1], hidden_dim=128, num_classes=len(label_list))
export_model.load_state_dict(torch.load(fp32_torch_path, map_location='cpu'))
export_model.eval()

dummy_input = torch.from_numpy(single_sample)

torch.onnx.export(
    export_model,
    dummy_input,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
)

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print('Saved ONNX model to', onnx_path)


In [ ]:

# Use CPUExecutionProvider because your current project plan is CPU-first.
# We keep this explicit so later measurements on Chameleon are easy to explain.

session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

onnx_session = ort.InferenceSession(
    str(onnx_path),
    sess_options=session_options,
    providers=['CPUExecutionProvider'],
)

onnx_metrics = {}
onnx_metrics.update(evaluate_onnx_session(onnx_session, X_test.astype(np.float32), y_test))
onnx_metrics.update(benchmark_onnx_latency(onnx_session, single_sample))
onnx_metrics.update(benchmark_onnx_throughput(onnx_session, batch_32))
onnx_metrics['artifact_size_mb'] = file_size_mb(onnx_path)
onnx_metrics['variant'] = 'onnxruntime_cpu_fp32'
print(json.dumps(onnx_metrics, indent=2))


## 8. Variant D — ONNX dynamic quantization (recommended quantization path)

In [ ]:

# Recommended quantization path for this notebook:
# ONNX dynamic quantization on CPU.
# Why this choice?
# - It is simple to reproduce on a CPU VM.
# - It tends to be lower risk than aggressive calibration-heavy approaches.
# - It often reduces model size for MLP-style graphs.
# - It matches the course idea of measuring tradeoffs instead of assuming a win.

quant_path = MODELS_DIR / 'tx_surrogate_dynamic_int8.onnx'
quantize_dynamic(
    model_input=str(onnx_path),
    model_output=str(quant_path),
    weight_type=QuantType.QInt8,
)
print('Saved quantized ONNX model to', quant_path)

quant_session = ort.InferenceSession(
    str(quant_path),
    providers=['CPUExecutionProvider'],
)

quant_metrics = {}
quant_metrics.update(evaluate_onnx_session(quant_session, X_test.astype(np.float32), y_test))
quant_metrics.update(benchmark_onnx_latency(quant_session, single_sample))
quant_metrics.update(benchmark_onnx_throughput(quant_session, batch_32))
quant_metrics['artifact_size_mb'] = file_size_mb(quant_path)
quant_metrics['variant'] = 'onnxruntime_cpu_dynamic_int8'
print(json.dumps(quant_metrics, indent=2))


## 9. Collect results into one comparison table

In [ ]:

rows = [eager_metrics, compiled_metrics, onnx_metrics, quant_metrics]
results_df = pd.DataFrame(rows)

# Reorder useful columns for easier reading.
preferred_cols = [
    'variant', 'status', 'top1_accuracy', 'top3_hit_rate',
    'p50_latency_ms', 'p95_latency_ms', 'throughput_items_per_sec',
    'artifact_size_mb'
]
for col in preferred_cols:
    if col not in results_df.columns:
        results_df[col] = np.nan
results_df = results_df[preferred_cols]

results_df


In [ ]:

# Save machine-readable outputs so they can later be copied into your serving options table.
summary_csv = RESULTS_DIR / 'model_optimization_summary.csv'
summary_json = RESULTS_DIR / 'model_optimization_summary.json'

results_df.to_csv(summary_csv, index=False)
with open(summary_json, 'w') as f:
    json.dump(rows, f, indent=2)

print('Saved CSV to', summary_csv)
print('Saved JSON to', summary_json)



## 10. What to do with this notebook next

After you reserve a CPU VM on Chameleon:

1. build the model-optimization container
2. run Jupyter inside the container
3. execute this notebook end-to-end on the VM
4. keep the executed notebook, saved artifacts, and summary CSV/JSON
5. use the best variant(s) as shortlist candidates for the later serving-system stage

A realistic interpretation for your project will likely look like this:

- **If ONNX or quantization wins clearly**: shortlist it for integration into the FastAPI serving service.
- **If eager or compiled is already fast enough**: keep the simpler CPU artifact and use that result to justify the cheaper design.
